## Training notebook

In [1]:
import gym
import highway_env
import datetime


# from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO

%load_ext tensorboard
import datetime, os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [3]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/training_output/'

In [4]:
# Create and wrap the environment
env = gym.make("dm-env-v0")

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [5]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [6]:
# PPO parameters
model = RecurrentPPO('MlpLstmPolicy', env, tensorboard_log=f'{OUTPUT_PATH}logdir', learning_rate=linear_schedule(3e-4), verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
num_timesteps = 2e4

# Train the agent for n steps
model.learn(total_timesteps=int(num_timesteps), callback=[checkpoint_callback, eval_callback])

#Save the trained agent
model.save(f'{OUTPUT_PATH}models/ppo_RL_{str(os_id.value)}_2e4_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')

Logging to C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/training_output/logdir\RecurrentPPO_4


: 

: 

### Continued learning

In [ ]:
env_cl = gym.make("jam-dm-env-v0")

In [ ]:
model_cl = PPO.load("/Users/fornerispighetti/HighwayDRL/training_output/models/ppo_RL_UNIX_4e4_8_6_2022_cl", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}logdir")

In [ ]:
# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=True)

In [ ]:
model_cl.learn(total_timesteps=int(2e5), reset_num_timesteps=False, callback=[checkpoint_callback, eval_callback])